# BTC-Vola: Ablauf & Auswertung

Die rechenintensiven Schritte laufen am besten **im Terminal** (siehe README), damit nichts abbricht, wenn der Browser zugeht. Dieses Notebook kannst du aber auch komplett von oben nach unten ausführen.

`MODE = 'valid'` → Tuning auf 2025. Erst wenn du mit den Einstellungen zufrieden bist, **einmal** `MODE = 'test'` (ab 2026).

In [ ]:
MODE = 'valid'   # 'valid' oder 'test'
import pandas as pd, numpy as np, matplotlib.pyplot as plt
pd.set_option('display.width', 200); pd.set_option('display.max_columns', 30)

## 1. Setup (einmalig)

In [ ]:
!pip install -q -r requirements.txt
import torch; print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'keine')

## 2. Daten laden & Features (ca. 5–15 Min.)

In [ ]:
!python s01_download.py --futures
!python s02_features.py

## 3. Baselines + LightGBM (CPU, ca. 10–40 Min.)

In [ ]:
!python s03_models.py --mode $MODE

## 4. Foundation Models (GPU)
Erst ein Schnelltest mit 20 Zeitpunkten, dann der volle Lauf.

In [ ]:
!python s04_foundation.py --mode $MODE --limit 20 --paths 8

In [ ]:
# Voller Lauf (kann je nach Modell 1–3 Std. dauern) – besser im Terminal starten:
# !python s04_foundation.py --mode $MODE --models chronos2 timestral kronos --every 6

## 5. Ergebnisse: Volatilität
QLIKE: niedriger = besser. `DM_p < 0.05` mit negativem `DM_t` = signifikant besser als HAR.

In [ ]:
import os
tag = f'foundation_{MODE}' if os.path.exists(f'results/vol_metrics_foundation_{MODE}.csv') else MODE
vt = pd.read_csv(f'results/vol_metrics_{tag}.csv')
display(vt.sort_values(['h','QLIKE']).round(4))
piv = vt.pivot(index='model', columns='h', values='QLIKE')
(piv / piv.loc['har']).T.plot(kind='bar', figsize=(10,4), title='QLIKE relativ zu HAR (<1 = besser)')
plt.axhline(1, color='k', lw=0.8); plt.ylabel('QLIKE / QLIKE(HAR)'); plt.show()

In [ ]:
vp = pd.read_parquet(f'results/vol_preds_{tag}.parquet')
h = 24
g = vp[vp.h == h].pivot_table(index='origin', columns='model', values='var')
y = vp[vp.h == h].groupby('origin')['y'].first()
ann = np.sqrt(365*24/h)   # annualisierte Vola
ax = (np.sqrt(y)*ann).rolling(24*7 if tag==MODE else 28).mean().plot(figsize=(12,4), color='k', lw=1.5, label='realisiert (geglättet)')
for m in [c for c in ['har','lgbm','chronos2','timestral_209M','kronos_small'] if c in g]:
    (np.sqrt(g[m])*ann).rolling(24*7 if tag==MODE else 28).mean().plot(ax=ax, lw=1, label=m)
ax.set_title(f'Vola-Prognose vs. realisiert, Horizont {h}h (annualisiert)'); ax.legend(); plt.show()

## 6. Ergebnisse: Touch-Wahrscheinlichkeiten
Das ist die Größe, die du später mit Polymarket-Preisen vergleichst. Wichtig ist vor allem **Kalibrierung**: Wenn das Modell 20 % sagt, sollte es in ~20 % der Fälle eintreten.

In [ ]:
tt = pd.read_csv(f'results/touch_metrics_{tag}.csv')
display(tt.pivot_table(index='model', columns=['h','dir'], values='Brier').round(4))
if 'BSS_vs_clim' in tt: display(tt.pivot_table(index='model', columns=['h','dir'], values='BSS_vs_clim').round(3))

In [ ]:
rel = pd.read_csv(f'results/reliability_{tag}.csv')
hs = sorted(rel.h.unique())
fig, axs = plt.subplots(1, len(hs), figsize=(4*len(hs), 4), sharey=True)
for ax, h in zip(np.atleast_1d(axs), hs):
    for m, g in rel[rel.h == h].groupby('model'):
        g = g[g.n >= 30]
        ax.plot(g.p_mean, g.hit_rate, 'o-', ms=3, label=m)
    ax.plot([0,1],[0,1],'k--',lw=0.8); ax.set_title(f'{h}h'); ax.set_xlabel('vorhergesagt')
np.atleast_1d(axs)[0].set_ylabel('eingetreten'); np.atleast_1d(axs)[-1].legend(fontsize=7)
plt.suptitle('Reliability-Diagramm (Diagonale = perfekt kalibriert)'); plt.tight_layout(); plt.show()

In [ ]:
# Kalibrierung je Schwelle: mittlere Prognose vs. tatsächliche Trefferquote
tp = pd.read_parquet(f'results/touch_preds_{tag}.parquet')
cal = tp.groupby(['h','dir','thr','model']).agg(prognose=('p','mean'), eingetreten=('hit','mean')).reset_index()
display(cal[cal.h == 24].pivot_table(index=['dir','thr'], columns='model', values='prognose').round(3)
        .join(cal[cal.h == 24].groupby(['dir','thr'])['eingetreten'].first().rename('TATSÄCHLICH').round(3)))

## 7. Nächste Schritte
- Features/Parameter **nur im Modus `valid`** verändern, `test` erst ganz am Ende einmal.
- Makro-Termine als `data/events.csv` ergänzen (Spalten `time,name`, Zeit in UTC) und `s02_features.py` neu laufen lassen.
- Danach: implizite Wahrscheinlichkeiten aus historischen Polymarket-Preisen gegen diese Prognosen stellen.